# Industrial Fire & Persistent Thermal Source Detection/Classification
### NASA FIRMS + OSM + ESA WorldCover Satellite Data

This notebook builds a machine-learning pipeline that classifies each FIRMS hotspot detection as either:

- **`vegetation_fire`** – a genuine wildland/crop-residue/forest fire (FIRMS `type = 0`)
- **`industrial_persistent`** – a persistent static thermal source such as a refinery flare, brick kiln, steel plant, gas flare, power plant, etc. (FIRMS `type = 2`, "other static land source")

We fuse three data sources, exactly matching the problem statement:

| Source | What it gives us |
|---|---|
| **NASA FIRMS** (VIIRS archive + NRT CSVs) | Brightness, FRP, day/night, confidence, and the ground-truth `type` label |
| **ESA WorldCover 10 m** (GeoTIFF) | Land-cover class at each hotspot (built-up vs cropland vs forest, etc.) |
| **OpenStreetMap (OSM)** | Distance to the nearest mapped industrial / power / mining polygon |

We also engineer a **persistence feature** (how many times a thermal source re-appears at ~the same spot over time) — this is the single strongest signal for "persistent thermal source" detection, since industrial sources burn continuously while vegetation fires are one-off events.

> ⚠️ **Only ONE cell below needs to be edited before running: the `CONFIG` cell.** Everything else runs as-is once your files are in place.

## 1. Install dependencies

Colab already has `pandas`, `numpy`, `scikit-learn`, `torch`. We additionally need `rasterio` (to read the GeoTIFF), `osmnx`/`geopandas`/`shapely` (for OSM), `dbfread` (for the FSI `.dbf` files) and `xgboost`.

In [ ]:
!pip install -q rasterio geopandas osmnx shapely dbfread xgboost folium

## 2. 📍 CONFIG — put your data path here

**This is the only cell you need to edit.**

1. Upload the whole dataset folder (all the `.csv`, `.dbf` and the `.tif` file) to your Google Drive, e.g. into a folder called `fire_data`.
2. Mount your Drive (next cell).
3. Set `DATA_DIR` below to the **folder that contains all the uploaded files**.

If you'd rather not use Drive, use the Colab file-upload widget instead (commented alternative below) and just set `DATA_DIR = "/content/"`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Your Google Drive folder shown in the screenshot
DATA_DIR = "/content/drive/MyDrive/SIH_DATASET"

import os

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"Folder not found: {DATA_DIR}")

print("DATA_DIR =", DATA_DIR)
print("\nAll dataset files found:")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files):
        print(f"{indent}  - {f}")


In [ ]:
# ================================================================
# AUTO-DETECT ALL DATA FILES INSIDE SIH_DATASET AND SUBFOLDERS
# No need to manually write exact filenames.
# ================================================================

import os

all_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        all_files.append(os.path.join(root, f))

csv_files = [p for p in all_files if p.lower().endswith(".csv")]
dbf_files = [p for p in all_files if p.lower().endswith(".dbf")]
tif_files = [p for p in all_files if p.lower().endswith((".tif", ".tiff"))]

# FIRMS archive files: SV = SNPP, J1V = NOAA-20
FIRMS_ARCHIVE_PATHS = [
    p for p in csv_files
    if ("SV-C2" in os.path.basename(p).upper()
        or "J1V-C2" in os.path.basename(p).upper()
        or "FIRE_ARCHIVE" in os.path.basename(p).upper())
]

# FIRMS NRT file: J2V = NOAA-21
nrt_candidates = [
    p for p in csv_files
    if ("J2V-C2" in os.path.basename(p).upper()
        or "FIRE_NRT" in os.path.basename(p).upper()
        or "_NRT_" in os.path.basename(p).upper())
]

FIRMS_NRT_PATH = nrt_candidates[0] if nrt_candidates else None

# All agricultural fire .dbf files
FSI_DBF_PATHS = dbf_files

# ESA WorldCover GeoTIFF
worldcover_candidates = [
    p for p in tif_files
    if "WORLDCOVER" in os.path.basename(p).upper()
]
WORLDCOVER_PATH = (
    worldcover_candidates[0]
    if worldcover_candidates else (tif_files[0] if tif_files else None)
)

print("========== DATA DETECTION ==========")
print("\nFIRMS ARCHIVE FILES:")
for p in FIRMS_ARCHIVE_PATHS:
    print(" -", p)

print("\nFIRMS NRT FILE:")
print(" -", FIRMS_NRT_PATH)

print("\nFSI DBF FILES:", len(FSI_DBF_PATHS))
for p in FSI_DBF_PATHS[:20]:
    print(" -", p)

print("\nWORLDCOVER FILE:")
print(" -", WORLDCOVER_PATH)

# Safety checks
if not FIRMS_ARCHIVE_PATHS:
    raise FileNotFoundError("No FIRMS archive CSV files were detected.")
if FIRMS_NRT_PATH is None:
    raise FileNotFoundError("No FIRMS NRT (J2V/NRT) CSV file was detected.")
if WORLDCOVER_PATH is None:
    raise FileNotFoundError("No WorldCover .tif file was detected.")


## 3. Load NASA FIRMS data

The **archive** files carry FIRMS' own `type` column — this is our ground-truth label:
`0 = vegetation fire`, `1 = active volcano`, `2 = other static land source (industrial/persistent thermal)`, `3 = offshore`.

The **NRT** file does not yet have `type` assigned (that's normal — FIRMS back-fills it later). We keep it aside and use our trained model to classify it at the end, which doubles as a live demo of the "detection" part of the problem statement.

In [ ]:
import pandas as pd
import numpy as np

def load_firms_archive(paths):
    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        df["source_file"] = os.path.basename(p)
        dfs.append(df)
    out = pd.concat(dfs, ignore_index=True)
    out["acq_datetime"] = pd.to_datetime(
        out["acq_date"] + " " + out["acq_time"].astype(str).str.zfill(4),
        format="%Y-%m-%d %H%M"
    )
    return out

firms_archive = load_firms_archive(FIRMS_ARCHIVE_PATHS)
print("Archive rows:", len(firms_archive))
print(firms_archive["type"].value_counts())
firms_archive.head()

In [ ]:
firms_nrt = pd.read_csv(FIRMS_NRT_PATH)
firms_nrt["source_file"] = os.path.basename(FIRMS_NRT_PATH)
firms_nrt["acq_datetime"] = pd.to_datetime(
    firms_nrt["acq_date"] + " " + firms_nrt["acq_time"].astype(str).str.zfill(4),
    format="%Y-%m-%d %H%M"
)
print("NRT rows (unlabeled):", len(firms_nrt))
firms_nrt.head()

## 4. (Optional) Load the FSI `.dbf` fire-location files

These are India's Forest Survey (FSI) fire alerts — a second, independent ground-truth source of **vegetation** fires (state/district/village level). We fold them in as extra `vegetation_fire` training examples, which helps balance the two classes (industrial `type=2` points are the minority class in FIRMS).

In [ ]:
from dbfread import DBF

def load_fsi_dbfs(paths):
    dfs = []
    for p in paths:
        try:
            table = DBF(p, load=True, encoding="latin1")
            df = pd.DataFrame(iter(table))
            df["source_file"] = os.path.basename(p)
            dfs.append(df)
        except Exception as e:
            print(f"Skipping {p}: {e}")
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

fsi = load_fsi_dbfs(FSI_DBF_PATHS)
print("FSI rows:", len(fsi))
if len(fsi):
    fsi.head()

In [ ]:
# Standardize FSI columns onto the FIRMS schema and label them as vegetation_fire
if len(fsi):
    fsi_std = pd.DataFrame({
        "latitude": pd.to_numeric(fsi["lat"], errors="coerce"),
        "longitude": pd.to_numeric(fsi["lon"], errors="coerce"),
        "brightness": pd.to_numeric(fsi["brightness"], errors="coerce"),
        "scan": pd.to_numeric(fsi["scanpixel_"], errors="coerce"),
        "track": pd.to_numeric(fsi["trackpixel"], errors="coerce"),
        "bright_t31": np.nan,
        "frp": pd.to_numeric(fsi["radiative_"], errors="coerce"),
        "confidence": "n",
        "daynight": "D",
        "satellite": fsi["sat"].astype(str),
        "acq_datetime": pd.to_datetime(fsi["acqdate"], errors="coerce"),
        "type": 0,                    # FSI = confirmed vegetation/forest fire alerts
        "source_file": fsi["source_file"],
    }).dropna(subset=["latitude", "longitude", "acq_datetime"])
else:
    fsi_std = pd.DataFrame()
print("Standardized FSI rows usable for training:", len(fsi_std))

## 5. Build the labeled training table

Combine FIRMS-archive + FSI into one table, keep only the two classes we care about (`0` vegetation, `2` industrial/persistent), and map them to a binary label.

In [ ]:
keep_cols = ["latitude", "longitude", "brightness", "scan", "track",
             "bright_t31", "frp", "confidence", "daynight", "satellite",
             "acq_datetime", "type", "source_file"]

archive_std = firms_archive[keep_cols].copy()
labeled = pd.concat([archive_std, fsi_std[keep_cols] if len(fsi_std) else archive_std.iloc[0:0]],
                     ignore_index=True)

labeled = labeled[labeled["type"].isin([0, 2])].reset_index(drop=True)
labeled["label"] = labeled["type"].map({0: "vegetation_fire", 2: "industrial_persistent"})

print(labeled["label"].value_counts())
print("Total labeled rows:", len(labeled))

## 6. Feature engineering

### 6.1 Persistence feature (the key signal)
We bin coordinates onto a ~1 km grid (`~0.01°`) and count, per grid cell, how many **distinct days** a thermal detection occurred. Industrial/persistent sources re-appear over many days; vegetation fires typically burn out in a few days.

In [ ]:
GRID = 0.01   # ~1.1 km at the equator; tune as needed

def add_persistence_features(df):
    df = df.copy()
    df["grid_lat"] = (df["latitude"] / GRID).round().astype(int)
    df["grid_lon"] = (df["longitude"] / GRID).round().astype(int)
    df["grid_cell"] = df["grid_lat"].astype(str) + "_" + df["grid_lon"].astype(str)

    persist = (df.groupby("grid_cell")["acq_datetime"]
                 .agg(detection_count="count",
                      distinct_days=lambda s: s.dt.date.nunique(),
                      first_seen="min",
                      last_seen="max"))
    persist["active_span_days"] = (persist["last_seen"] - persist["first_seen"]).dt.days + 1

    df = df.merge(persist, on="grid_cell", how="left")
    return df

labeled = add_persistence_features(labeled)
labeled[["grid_cell", "detection_count", "distinct_days", "active_span_days"]].describe()

### 6.2 Land-cover feature (ESA WorldCover, 10 m)

We sample the WorldCover raster at each hotspot's coordinates. WorldCover classes (subset relevant here): `50 = Built-up`, `40 = Cropland`, `10 = Tree cover`, `30 = Grassland`, `20 = Shrubland`, `60 = Bare/sparse vegetation`, `80 = Permanent water bodies`.

> **Note:** the single tile you have (`N27E075`) only covers a ~1°×1° area (roughly around 27°N, 75°E). Points outside that tile will get `NaN` land cover — the model still trains fine (missing-value handling is built into the pipeline), but for full national coverage you'd download the neighbouring WorldCover tiles from https://esa-worldcover.org and add them to `DATA_DIR`/loop over them the same way.

In [ ]:
import rasterio
from rasterio.transform import rowcol

def sample_landcover(df, tif_path):
    df = df.copy()
    df["landcover"] = np.nan
    with rasterio.open(tif_path) as src:
        band = src.read(1)
        left, bottom, right, top = src.bounds
        in_bounds = (df["longitude"].between(left, right)) & (df["latitude"].between(bottom, top))
        sub = df[in_bounds]
        for idx, row_ in sub.iterrows():
            try:
                r, c = rowcol(src.transform, row_["longitude"], row_["latitude"])
                if 0 <= r < band.shape[0] and 0 <= c < band.shape[1]:
                    df.at[idx, "landcover"] = band[r, c]
            except Exception:
                pass
    return df

labeled = sample_landcover(labeled, WORLDCOVER_PATH)
print("Points with a land-cover value:", labeled["landcover"].notna().sum(), "/", len(labeled))
labeled["landcover"].value_counts(dropna=False)

### 6.3 OSM feature — distance to nearest industrial / power / mining area

We query OpenStreetMap (via `osmnx`) for `landuse=industrial`, `power=plant`, `landuse=quarry` and `man_made=works` polygons inside the bounding box of your data, then compute each hotspot's distance to the nearest one. Being **inside or very close to** a mapped industrial polygon is strong corroborating evidence for `industrial_persistent`.

> This step needs internet access (fine on Colab) and can take a few minutes for a large bounding box — it only needs to run once, so we cache the result to a GeoJSON.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, box
import osmnx as ox

osm_cache_path = os.path.join(DATA_DIR, "osm_industrial_cache.geojson")

def get_osm_industrial(bbox, cache_path):
    if os.path.exists(cache_path):
        print("Loading cached OSM industrial features...")
        return gpd.read_file(cache_path)

    north, south, east, west = bbox
    tags = {"landuse": ["industrial", "quarry"], "power": ["plant", "generator"],
            "man_made": ["works", "kiln"]}
    print("Querying OSM (this can take a few minutes for a large area)...")
    gdf = ox.features_from_bbox((north, south, east, west), tags)
    gdf = gdf[gdf.geometry.notna()].to_crs(epsg=4326)
    gdf.to_file(cache_path, driver="GeoJSON")
    return gdf

pad = 0.5  # degrees of padding around your data extent
bbox = (labeled["latitude"].max() + pad, labeled["latitude"].min() - pad,
        labeled["longitude"].max() + pad, labeled["longitude"].min() - pad)

osm_industrial = get_osm_industrial(bbox, osm_cache_path)
print("OSM industrial features found:", len(osm_industrial))

In [ ]:
def add_osm_distance(df, osm_gdf, batch_size=20000):
    df = df.copy()
    if osm_gdf is None or len(osm_gdf) == 0:
        df["osm_industrial_dist_km"] = np.nan
        df["osm_inside_industrial"] = 0
        return df

    industrial_union = osm_gdf.geometry.unary_union
    pts = gpd.GeoSeries([Point(xy) for xy in zip(df["longitude"], df["latitude"])], crs="EPSG:4326")

    # project to a metric CRS (meters) for accurate distances
    pts_m = pts.to_crs(epsg=3857)
    industrial_m = gpd.GeoSeries([industrial_union], crs="EPSG:4326").to_crs(epsg=3857).iloc[0]

    dist_m = pts_m.distance(industrial_m)
    df["osm_industrial_dist_km"] = dist_m.values / 1000.0
    df["osm_inside_industrial"] = (dist_m.values == 0).astype(int)
    return df

labeled = add_osm_distance(labeled, osm_industrial)
labeled[["osm_industrial_dist_km", "osm_inside_industrial"]].describe()

## 7. Final feature matrix + train/test split

In [ ]:
FEATURES_NUM = ["brightness", "scan", "track", "bright_t31", "frp",
                "detection_count", "distinct_days", "active_span_days",
                "landcover", "osm_industrial_dist_km", "osm_inside_industrial"]
FEATURES_CAT = ["confidence", "daynight", "satellite"]

model_df = labeled.dropna(subset=["brightness", "frp"]).reset_index(drop=True)

X = model_df[FEATURES_NUM + FEATURES_CAT]
y = model_df["label"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print(y_train.value_counts(normalize=True))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, FEATURES_NUM),
    ("cat", categorical_pipe, FEATURES_CAT),
])

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
print(dict(zip(le.classes_, range(len(le.classes_)))))

## 8. Baseline models: Random Forest & XGBoost

Strong, fast, interpretable baselines for tabular data — good sanity checks before the neural net.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

Xtr = preprocessor.fit_transform(X_train)
Xte = preprocessor.transform(X_test)

rf = RandomForestClassifier(
    n_estimators=400, max_depth=None, class_weight="balanced",
    n_jobs=-1, random_state=42
)
rf.fit(Xtr, y_train_enc)
rf_pred = rf.predict(Xte)
print("=== Random Forest ===")
print(classification_report(y_test_enc, rf_pred, target_names=le.classes_))
print("ROC-AUC:", roc_auc_score(y_test_enc, rf.predict_proba(Xte)[:, 1]))

In [ ]:
pos_weight = (y_train_enc == 0).sum() / (y_train_enc == 1).sum()

xgb = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=pos_weight, eval_metric="logloss",
    random_state=42, n_jobs=-1
)
xgb.fit(Xtr, y_train_enc)
xgb_pred = xgb.predict(Xte)
print("=== XGBoost ===")
print(classification_report(y_test_enc, xgb_pred, target_names=le.classes_))
print("ROC-AUC:", roc_auc_score(y_test_enc, xgb.predict_proba(Xte)[:, 1]))

## 9. Deep-learning model (PyTorch MLP)

This is the core "AI model" deliverable — a feed-forward neural network trained on the same fused FIRMS + WorldCover + OSM features, with class-weighted loss to handle the imbalance between the two classes.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Xtr_t = torch.tensor(Xtr.toarray() if hasattr(Xtr, "toarray") else Xtr, dtype=torch.float32)
Xte_t = torch.tensor(Xte.toarray() if hasattr(Xte, "toarray") else Xte, dtype=torch.float32)
ytr_t = torch.tensor(y_train_enc, dtype=torch.long)
yte_t = torch.tensor(y_test_enc, dtype=torch.long)

train_ds = TensorDataset(Xtr_t, ytr_t)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

n_features = Xtr_t.shape[1]

class FireMLP(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64, 32), n_classes=2, dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, n_classes)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = FireMLP(n_features).to(device)

class_counts = np.bincount(y_train_enc)
class_weights = torch.tensor(len(y_train_enc) / (len(class_counts) * class_counts), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

In [ ]:
EPOCHS = 30
Xte_dev, yte_dev = Xte_t.to(device), yte_t.to(device)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    model.eval()
    with torch.no_grad():
        val_out = model(Xte_dev)
        val_loss = criterion(val_out, yte_dev).item()
        val_acc = (val_out.argmax(1) == yte_dev).float().mean().item()
    scheduler.step(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train_loss {total_loss/len(train_ds):.4f} "
              f"| val_loss {val_loss:.4f} | val_acc {val_acc:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(Xte_dev)
    probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    mlp_pred = logits.argmax(1).cpu().numpy()

print("=== PyTorch MLP ===")
print(classification_report(y_test_enc, mlp_pred, target_names=le.classes_))
print("ROC-AUC:", roc_auc_score(y_test_enc, probs))
print("Confusion matrix:\n", confusion_matrix(y_test_enc, mlp_pred))

## 10. Feature importance (Random Forest) & model comparison

In [ ]:
feat_names = (FEATURES_NUM +
              list(preprocessor.named_transformers_["cat"]
                   .named_steps["onehot"].get_feature_names_out(FEATURES_CAT)))

importances = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
print(importances.head(15))

import matplotlib.pyplot as plt
importances.head(15).sort_values().plot(kind="barh", figsize=(8, 6), title="Random Forest feature importance")
plt.tight_layout()
plt.show()

## 11. Save the trained model + preprocessing pipeline

In [ ]:
import joblib

SAVE_DIR = os.path.join(DATA_DIR, "model_output")
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(preprocessor, os.path.join(SAVE_DIR, "preprocessor.joblib"))
joblib.dump(le, os.path.join(SAVE_DIR, "label_encoder.joblib"))
joblib.dump(rf, os.path.join(SAVE_DIR, "random_forest.joblib"))
xgb.save_model(os.path.join(SAVE_DIR, "xgboost_model.json"))
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "fire_mlp.pt"))

print("Saved all artifacts to:", SAVE_DIR)

## 12. Apply the trained model to the unlabeled NRT feed (live detection demo)

This is the "detection" half of the problem statement: take fresh, not-yet-classified FIRMS NRT hotspots and label each one as `vegetation_fire` or `industrial_persistent` using the model we just trained.

In [ ]:
nrt_feat = firms_nrt.copy()
nrt_feat = add_persistence_features(
    pd.concat([labeled[["latitude", "longitude", "acq_datetime"]], nrt_feat[["latitude", "longitude", "acq_datetime"]]],
              ignore_index=True)
).tail(len(nrt_feat)).reset_index(drop=True)   # recompute persistence using full history + new points

nrt_feat = sample_landcover(nrt_feat, WORLDCOVER_PATH)
nrt_feat = add_osm_distance(nrt_feat, osm_industrial)

X_nrt = nrt_feat.reindex(columns=FEATURES_NUM + FEATURES_CAT)
X_nrt_proc = preprocessor.transform(X_nrt)

with torch.no_grad():
    x_t = torch.tensor(X_nrt_proc.toarray() if hasattr(X_nrt_proc, "toarray") else X_nrt_proc, dtype=torch.float32).to(device)
    nrt_logits = model(x_t)
    nrt_pred = le.inverse_transform(nrt_logits.argmax(1).cpu().numpy())
    nrt_conf = torch.softmax(nrt_logits, dim=1).max(1).values.cpu().numpy()

nrt_feat["predicted_label"] = nrt_pred
nrt_feat["prediction_confidence"] = nrt_conf

out_path = os.path.join(SAVE_DIR, "nrt_classified.csv")
nrt_feat.to_csv(out_path, index=False)
print("Classified NRT detections saved to:", out_path)
nrt_feat[["latitude", "longitude", "acq_datetime", "frp", "predicted_label", "prediction_confidence"]].head(20)

## 13. (Optional) Map the classified points

In [ ]:
import folium

sample = nrt_feat.sample(min(2000, len(nrt_feat)), random_state=1)
center = [sample["latitude"].mean(), sample["longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="CartoDB positron")

colors = {"vegetation_fire": "green", "industrial_persistent": "red"}
for _, r in sample.iterrows():
    folium.CircleMarker(
        location=[r["latitude"], r["longitude"]],
        radius=3, color=colors.get(r["predicted_label"], "gray"),
        fill=True, fill_opacity=0.7,
        popup=f"{r['predicted_label']} ({r['prediction_confidence']:.2f})"
    ).add_to(m)

m

## Notes & next steps

- **Land-cover coverage**: only the single WorldCover tile `N27E075` was provided, so points outside that ~1°×1° tile get `NaN` land cover (handled via median-imputation, but accuracy improves with full coverage). Download neighbouring tiles from the [ESA WorldCover viewer](https://esa-worldcover.org/en) and extend `sample_landcover` to loop over multiple tiles (merge with `rasterio.merge`).
- **OSM completeness**: OSM industrial-polygon tagging is crowd-sourced and incomplete in some regions — treat `osm_industrial_dist_km` as corroborating evidence, not ground truth.
- **Persistence window**: `GRID` and the day-count window can be tuned; for a stricter "persistent thermal source" definition, require e.g. `distinct_days >= 10` within a rolling 90-day window rather than the whole archive.
- **Class imbalance**: both `class_weight="balanced"` (RF), `scale_pos_weight` (XGBoost) and a weighted `CrossEntropyLoss` (MLP) are already applied — re-check the confusion matrices if you significantly change the dataset.
- **Multiclass extension**: to also separately flag volcanoes (`type=1`) and offshore flares (`type=3`) instead of dropping them, just include them in step 5's `.isin([...])` filter and switch `LabelEncoder`/loss to handle 4 classes.